# 基于截断策略的机器阅读理解任务实现

## Step1 导入相关包

In [36]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)

## Step2 数据集加载

In [37]:
datasets = load_dataset("hfl/cmrc2018", cache_dir="./mrc_data/")
datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 10142
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 3219
    })
    test: Dataset({
        features: ['id', 'context', 'question', 'answers'],
        num_rows: 1002
    })
})

In [38]:
datasets["train"][0]

{'id': 'TRAIN_186_QUERY_0',
 'context': '范廷颂枢机（，），圣名保禄·若瑟（），是越南罗马天主教枢机。1963年被任为主教；1990年被擢升为天主教河内总教区宗座署理；1994年被擢升为总主教，同年年底被擢升为枢机；2009年2月离世。范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生；童年时接受良好教育后，被一位越南神父带到河内继续其学业。范廷颂于1940年在河内大修道院完成神学学业。范廷颂于1949年6月6日在河内的主教座堂晋铎；及后被派到圣女小德兰孤儿院服务。1950年代，范廷颂在河内堂区创建移民接待中心以收容到河内避战的难民。1954年，法越战争结束，越南民主共和国建都河内，当时很多天主教神职人员逃至越南的南方，但范廷颂仍然留在河内。翌年管理圣若望小修院；惟在1960年因捍卫修院的自由、自治及拒绝政府在修院设政治课的要求而被捕。1963年4月5日，教宗任命范廷颂为天主教北宁教区主教，同年8月15日就任；其牧铭为「我信天主的爱」。由于范廷颂被越南政府软禁差不多30年，因此他无法到所属堂区进行牧灵工作而专注研读等工作。范廷颂除了面对战争、贫困、被当局迫害天主教会等问题外，也秘密恢复修院、创建女修会团体等。1990年，教宗若望保禄二世在同年6月18日擢升范廷颂为天主教河内总教区宗座署理以填补该教区总主教的空缺。1994年3月23日，范廷颂被教宗若望保禄二世擢升为天主教河内总教区总主教并兼天主教谅山教区宗座署理；同年11月26日，若望保禄二世擢升范廷颂为枢机。范廷颂在1995年至2001年期间出任天主教越南主教团主席。2003年4月26日，教宗若望保禄二世任命天主教谅山教区兼天主教高平教区吴光杰主教为天主教河内总教区署理主教；及至2005年2月19日，范廷颂因获批辞去总主教职务而荣休；吴光杰同日真除天主教河内总教区总主教职务。范廷颂于2009年2月22日清晨在河内离世，享年89岁；其葬礼于同月26日上午在天主教河内总教区总主教座堂举行。',
 'question': '范廷颂是什么时候被任为主教的？',
 'answers': {'text': ['1963年'], 'answer_start': [30]}}

In [39]:
datasets["train"].features

{'id': Value('string'),
 'context': Value('string'),
 'question': Value('string'),
 'answers': {'text': List(Value('string')),
  'answer_start': List(Value('int32'))}}

## Step3 数据预处理

In [40]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")
tokenizer

BertTokenizer(name_or_path='hfl/chinese-macbert-base', vocab_size=21128, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [41]:
sample_dateset = datasets["train"].select(range(10))
sample_dateset

Dataset({
    features: ['id', 'context', 'question', 'answers'],
    num_rows: 10
})

In [42]:
tokenizered_examples = tokenizer(
    text=list(sample_dateset["question"]),
    text_pair=list(sample_dateset["context"]),
    max_length=384,
    truncation="only_second",
    padding="max_length",
    return_offsets_mapping=True,
    return_overflowing_tokens=True, # 新增参数，返回overflowing tokens的映射关系
    stride=128, # 新增参数，设置滑动窗口的步长
)

list(tokenizered_examples.keys())

['input_ids',
 'token_type_ids',
 'attention_mask',
 'offset_mapping',
 'overflow_to_sample_mapping']

In [43]:
print(tokenizered_examples["token_type_ids"][0])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [44]:
tokenizered_examples["overflow_to_sample_mapping"],len(tokenizered_examples["overflow_to_sample_mapping"])

([0,
  0,
  0,
  1,
  1,
  1,
  2,
  2,
  2,
  3,
  3,
  3,
  4,
  4,
  4,
  5,
  5,
  5,
  6,
  6,
  6,
  7,
  7,
  7,
  8,
  8,
  8,
  9,
  9],
 29)

In [45]:
for sen in tokenizer.batch_decode(tokenizered_examples["input_ids"][:3]):
    print(sen,"\n")

[CLS] 范 廷 颂 是 什 么 时 候 被 任 为 主 教 的 ？ [SEP] 范 廷 颂 枢 机 （ ， ） ， 圣 名 保 禄 · 若 瑟 （ ） ， 是 越 南 罗 马 天 主 教 枢 机 。 1963 年 被 任 为 主 教 ； 1990 年 被 擢 升 为 天 主 教 河 内 总 教 区 宗 座 署 理 ； 1994 年 被 擢 升 为 总 主 教 ， 同 年 年 底 被 擢 升 为 枢 机 ； 2009 年 2 月 离 世 。 范 廷 颂 于 1919 年 6 月 15 日 在 越 南 宁 平 省 天 主 教 发 艳 教 区 出 生 ； 童 年 时 接 受 良 好 教 育 后 ， 被 一 位 越 南 神 父 带 到 河 内 继 续 其 学 业 。 范 廷 颂 于 1940 年 在 河 内 大 修 道 院 完 成 神 学 学 业 。 范 廷 颂 于 1949 年 6 月 6 日 在 河 内 的 主 教 座 堂 晋 铎 ； 及 后 被 派 到 圣 女 小 德 兰 孤 儿 院 服 务 。 1950 年 代 ， 范 廷 颂 在 河 内 堂 区 创 建 移 民 接 待 中 心 以 收 容 到 河 内 避 战 的 难 民 。 1954 年 ， 法 越 战 争 结 束 ， 越 南 民 主 共 和 国 建 都 河 内 ， 当 时 很 多 天 主 教 神 职 人 员 逃 至 越 南 的 南 方 ， 但 范 廷 颂 仍 然 留 在 河 内 。 翌 年 管 理 圣 若 望 小 修 院 ； 惟 在 1960 年 因 捍 卫 修 院 的 自 由 、 自 治 及 拒 绝 政 府 在 修 院 设 政 治 课 的 要 求 而 被 捕 。 1963 年 4 月 5 日 ， 教 宗 任 命 范 廷 颂 为 天 主 教 北 宁 教 区 主 教 ， 同 年 8 月 15 日 就 任 ； 其 牧 铭 为 「 我 信 [SEP] 

[CLS] 范 廷 颂 是 什 么 时 候 被 任 为 主 教 的 ？ [SEP] 越 南 民 主 共 和 国 建 都 河 内 ， 当 时 很 多 天 主 教 神 职 人 员 逃 至 越 南 的 南 方 ， 但 范 廷 颂 仍 然 留 在 河 内 。 翌 年 管 理 圣 若 望 小 修 院 ； 惟 在 1960 年 因 捍 卫 修 院 的 自 由 、 自 治 及 

In [46]:
help(tokenizered_examples)

Help on BatchEncoding in module transformers.tokenization_utils_base object:

class BatchEncoding(collections.UserDict, typing.Generic)
 |  BatchEncoding(
 |      data: dict[str, Any] | None = None,
 |      encoding: EncodingFast | Sequence[EncodingFast] | None = None,
 |      tensor_type: None | str | TensorType = None,
 |      prepend_batch_axis: bool = False,
 |      n_sequences: int | None = None
 |  )
 |
 |  Holds the output of the [`~tokenization_utils_base.PreTrainedTokenizerBase.__call__`],
 |  [`~tokenization_utils_base.PreTrainedTokenizerBase.encode_plus`] and
 |  [`~tokenization_utils_base.PreTrainedTokenizerBase.batch_encode_plus`] methods (tokens, attention_masks, etc).
 |
 |  This class is derived from a python dictionary and can be used as a dictionary. In addition, this class exposes
 |  utility methods to map from word/character space to token space.
 |
 |  Args:
 |      data (`dict`, *optional*):
 |          Dictionary of lists/arrays/tensors returned by the `__call__

In [47]:
print(tokenizered_examples.sequence_ids())

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [48]:
print(tokenizered_examples["offset_mapping"][:3])
# 此时offset_mapping 中第2、3个列表中context部分的起始offset不再是0，而是相对于原始context的offset，这样我们就可以根据overflow_to_sample_mapping来找到对应的原始context，从而正确地计算答案的起始和结束位置。

[[(0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16), (16, 17), (17, 18), (18, 19), (19, 20), (20, 21), (21, 22), (22, 23), (23, 24), (24, 25), (25, 26), (26, 27), (27, 28), (28, 29), (29, 30), (30, 34), (34, 35), (35, 36), (36, 37), (37, 38), (38, 39), (39, 40), (40, 41), (41, 45), (45, 46), (46, 47), (47, 48), (48, 49), (49, 50), (50, 51), (51, 52), (52, 53), (53, 54), (54, 55), (55, 56), (56, 57), (57, 58), (58, 59), (59, 60), (60, 61), (61, 62), (62, 63), (63, 67), (67, 68), (68, 69), (69, 70), (70, 71), (71, 72), (72, 73), (73, 74), (74, 75), (75, 76), (76, 77), (77, 78), (78, 79), (79, 80), (80, 81), (81, 82), (82, 83), (83, 84), (84, 85), (85, 86), (86, 87), (87, 91), (91, 92), (92, 93), (93, 94), (94, 95), (95, 96), (96, 97), (97, 98), (98, 99), 

In [49]:
list(tokenizered_examples.keys())

['input_ids',
 'token_type_ids',
 'attention_mask',
 'offset_mapping',
 'overflow_to_sample_mapping']

In [50]:
offset_mapping = tokenizered_examples.pop("offset_mapping")
sample_mapping = tokenizered_examples.pop("overflow_to_sample_mapping")

In [51]:
for batch_idx, _ in enumerate(sample_mapping):
    answer = sample_dateset["answers"][sample_mapping[batch_idx]]
    start_answer = answer["answer_start"][0]
    end_answer = start_answer + len(answer["text"][0])

    # context的tokens在inputs_ids中的起始位置
    start_context = tokenizered_examples.sequence_ids(batch_idx).index(1)
    end_context = (
        tokenizered_examples.sequence_ids(batch_idx).index(None, start_context) - 1
    )

    # 判断answer是否在context中,如果不在,则设置start_token和end_token为0
    # 如果在,则找到answer在context的tokens中的起始位置和结束位置
    offset = offset_mapping[batch_idx]
    if offset[start_context][0] > end_answer or offset[end_context][1] < start_answer:
        start_token = 0
        end_token = 0
    else:
        start_token = start_context
        while start_token <= end_context and offset[start_token][0] < start_answer:
            start_token += 1
        end_token = end_context
        while end_token >= start_context and offset[end_token][1] > end_answer:
            end_token -= 1
    print(
        f"answer: {answer['text'][0]}",
        f"start_context: {start_context}",
        f"end_context: {end_context}",
        f"start_token: {start_token}",
        f"end_token: {end_token}",
    )
    print(f"decoded answer: {tokenizer.decode(tokenizered_examples['input_ids'][batch_idx][start_token:end_token+1])}")

answer: 1963年 start_context: 17 end_context: 382 start_token: 47 end_token: 48
decoded answer: 1963 年
answer: 1963年 start_context: 17 end_context: 382 start_token: 0 end_token: 0
decoded answer: [CLS]
answer: 1963年 start_context: 17 end_context: 289 start_token: 0 end_token: 0
decoded answer: [CLS]
answer: 1990年被擢升为天主教河内总教区宗座署理 start_context: 15 end_context: 382 start_token: 53 end_token: 70
decoded answer: 1990 年 被 擢 升 为 天 主 教 河 内 总 教 区 宗 座 署 理
answer: 1990年被擢升为天主教河内总教区宗座署理 start_context: 15 end_context: 382 start_token: 0 end_token: 0
decoded answer: [CLS]
answer: 1990年被擢升为天主教河内总教区宗座署理 start_context: 15 end_context: 283 start_token: 0 end_token: 0
decoded answer: [CLS]
answer: 范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生 start_context: 15 end_context: 382 start_token: 100 end_token: 124
decoded answer: 范 廷 颂 于 1919 年 6 月 15 日 在 越 南 宁 平 省 天 主 教 发 艳 教 区 出 生
answer: 范廷颂于1919年6月15日在越南宁平省天主教发艳教区出生 start_context: 15 end_context: 382 start_token: 0 end_token: 0
decoded answer: [CLS]
answer: 范廷颂于1919年6月15日

In [52]:
def process_func(examples):
    tokenizered_examples = tokenizer(
        text=list(examples["question"]),
        text_pair=list(examples["context"]),
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
        return_overflowing_tokens=True, # 新增参数，返回overflowing tokens的映射关系
        stride=128, # 新增参数，设置滑动窗口的步长
    )

    start_positions = []
    end_positions = []
    examples_ids = []

    sample_mapping = tokenizered_examples.pop("overflow_to_sample_mapping")
    for batch_idx, _ in enumerate(sample_mapping):
        # 这里函数传入的examples是一个字典，而不是一个Datasets，所以没办法首先通过batch_idx索引获取batch
        # 只能通过examples["answers"]来获取所有batch的answers,然后通过batch_idx来获取当前batch的answer
        answer = examples["answers"][sample_mapping[batch_idx]]
        start_answer = answer["answer_start"][0]
        end_answer = start_answer + len(answer["text"][0])

        # context的tokens在inputs_ids中的起始位置
        start_context = tokenizered_examples.sequence_ids(batch_idx).index(1)
        end_context = (
            tokenizered_examples.sequence_ids(batch_idx).index(None, start_context) - 1
        )

        offset = tokenizered_examples["offset_mapping"][batch_idx]

        # 判断answer是否在context中,如果不在,则设置start_token和end_token为0
        # 如果在,则找到answer在context的tokens中的起始位置和结束位置
        if (
            offset[start_context][0] > end_answer
            or offset[end_context][1] < start_answer
        ):
            start_token = 0
            end_token = 0
        else:
            start_token = start_context
            while start_token <= end_context and offset[start_token][0] < start_answer:
                start_token += 1
            end_token = end_context
            while end_token >= start_context and offset[end_token][1] > end_answer:
                end_token -= 1
        start_positions.append(start_token)
        end_positions.append(end_token)
        examples_ids.append(examples["id"][sample_mapping[batch_idx]])
        tokenizered_examples["offset_mapping"][batch_idx] = [
            o if tokenizered_examples.sequence_ids(batch_idx)[k] == 1 else None
            for k,o in enumerate(tokenizered_examples["offset_mapping"][batch_idx])
        ]

    tokenizered_examples["start_positions"] = start_positions
    tokenizered_examples["end_positions"] = end_positions
    tokenizered_examples["examples_ids"] = examples_ids
    return tokenizered_examples

In [53]:
tokenized_datasets = datasets.map(
    process_func, batched=True, remove_columns=datasets["train"].column_names
)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'examples_ids'],
        num_rows: 19189
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'examples_ids'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions', 'examples_ids'],
        num_rows: 1988
    })
})

In [54]:
print(tokenized_datasets["train"]["offset_mapping"][0])

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, [0, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9], [9, 10], [10, 11], [11, 12], [12, 13], [13, 14], [14, 15], [15, 16], [16, 17], [17, 18], [18, 19], [19, 20], [20, 21], [21, 22], [22, 23], [23, 24], [24, 25], [25, 26], [26, 27], [27, 28], [28, 29], [29, 30], [30, 34], [34, 35], [35, 36], [36, 37], [37, 38], [38, 39], [39, 40], [40, 41], [41, 45], [45, 46], [46, 47], [47, 48], [48, 49], [49, 50], [50, 51], [51, 52], [52, 53], [53, 54], [54, 55], [55, 56], [56, 57], [57, 58], [58, 59], [59, 60], [60, 61], [61, 62], [62, 63], [63, 67], [67, 68], [68, 69], [69, 70], [70, 71], [71, 72], [72, 73], [73, 74], [74, 75], [75, 76], [76, 77], [77, 78], [78, 79], [79, 80], [80, 81], [81, 82], [82, 83], [83, 84], [84, 85], [85, 86], [86, 87], [87, 91], [91, 92], [92, 93], [93, 94], [94, 95], [95, 96], [96, 97], [97, 98], [98, 99], [99, 100], [100, 101], [101, 105], [105, 106],

## Step4 获取模型输出

In [55]:
import collections

# example 和 feature的映射
example_to_feature = collections.defaultdict(list)
for idx, example_id in enumerate(tokenized_datasets["train"]["examples_ids"][:10]):
    example_to_feature[example_id].append(idx)
example_to_feature

defaultdict(list,
            {'TRAIN_186_QUERY_0': [0, 1, 2],
             'TRAIN_186_QUERY_1': [3, 4, 5],
             'TRAIN_186_QUERY_2': [6, 7, 8],
             'TRAIN_186_QUERY_3': [9]})

In [56]:
import itertools
import numpy as np
from collections import defaultdict


def get_results(start_logits, end_logits, examples, features):
    """
    根据模型输出的start_logits和end_logits，结合examples和features的信息，计算最终的答案文本。
    Args:
     - start_logits: 模型输出的start_logits，形状为(batch_size, sequence_length)
     - end_logits: 模型输出的end_logits，形状为(batch_size, sequence_length)
     - examples: 原始的examples数据集，包含问题、上下文和答案等信息
     - features: 经过tokenizer处理后的features数据集，包含input_ids、offset_mapping等信息
    """
    predictions = {}
    references = {}

    # 建立example和feature的映射
    example_to_feature = defaultdict(list)
    for idx, example_id in enumerate(features["examples_ids"]):
        example_to_feature[example_id].append(idx)

    # 最优答案候选列表长度
    n_best = 20
    # 最大答案长度
    max_answer_length = 30

    for example in examples:
        example_id = example["id"]
        context = example["context"]
        answers = []

        for feature_idx in example_to_feature[example_id]:
            start_logit = start_logits[feature_idx]
            end_logit = end_logits[feature_idx]
            offset_mapping = features["offset_mapping"][feature_idx]
            start_logits_indices = np.argsort(start_logit)[::-1][:n_best].tolist()
            end_logits_indices = np.argsort(end_logit)[::-1][:n_best].tolist()
            for start_index, end_index in itertools.product(
                start_logits_indices, end_logits_indices
            ):
                if (
                    offset_mapping[start_index] is None
                    or offset_mapping[end_index] is None
                ):
                    continue
                if (
                    end_index < start_index
                    or end_index - start_index + 1 > max_answer_length
                ):
                    continue
                answers.append({
                    "text":context[offset_mapping[start_index][0] : offset_mapping[end_index][1]],
                    "score": start_logit[start_index] + end_logit[end_index]
                })
        if len(answers)>0:
            best_answer = max(answers, key=lambda x: x["score"])
            predictions[example_id] = best_answer["text"]
        else:
            predictions[example_id] = ""
        references[example_id] = example["answers"]["text"]

    return predictions, references


## Step5 评估函数

In [57]:
from cmrc_eval import evaluate_cmrc

def metric(pred):
    start_logits, end_logits = pred[0]

    if start_logits.shape[0] == len(tokenized_datasets["validation"]):
        p,r = get_results(start_logits, end_logits, datasets["validation"], tokenized_datasets["validation"])
    else:
        p, r = get_results(start_logits, end_logits, datasets["test"], tokenized_datasets["test"])
    return evaluate_cmrc(p, r)

## Step6 加载模型

In [58]:
model =AutoModelForQuestionAnswering.from_pretrained("hfl/chinese-macbert-base")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ign

## Step7 配置训练参数

In [59]:
args = TrainingArguments(
    output_dir="./models_for_qa",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_steps=200,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=3
)

## Step8 配置trainer

In [60]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DefaultDataCollator(),
    compute_metrics=metric
)

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "d:\code_local\Python\transformers-code\.venv\Lib\site-packages\huggingface_hub\utils\_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "d:\code_local\Python\transformers-code\.venv\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/chinese-macbert-base/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\YYH\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File 

## Step9 训练模型

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Step10 模型预测

In [ ]:
import torch

def question_answering(question, context, model, tokenizer, device=0):
    """手动实现问答推理，替代已被移除的 question-answering pipeline"""
    model.eval()
    model = model.to(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
    
    inputs = tokenizer(
        question, context,
        return_tensors="pt",
        max_length=384,
        truncation="only_second",
        padding="max_length",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    
    answer_ids = inputs["input_ids"][0][start_idx:end_idx + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
    score = (
        torch.softmax(outputs.start_logits, dim=-1)[0][start_idx].item()
        + torch.softmax(outputs.end_logits, dim=-1)[0][end_idx].item()
    ) / 2
    
    return {"answer": answer, "score": score, "start": start_idx.item(), "end": end_idx.item()}

# 使用示例
question = "中国的首都是哪里？"
context = "中华人民共和国首都是北京，位于华北平原北部。"
result = question_answering(question, context, model, tokenizer)
print(result)